# DiffuGPT-S: Layer-averaged attention entropy over diffusion time

Modifies Raghu's Issue #47 entropy code so that, instead of one cherry-picked head, the entropy is **averaged across all heads in a layer**. For each diffusion step we compute every head's per-token attention entropy and average over heads, giving a per-layer entropy signal.

Goal: show that **Layer 0 entropy increases** and **Layer 10 entropy decreases** over diffusion time, as a layer-level (not head-level) effect.

Two views are produced, both saved as **Type 1 / embedded-font PDFs** (`plt.rcParams['pdf.fonttype'] = 42`):
1. Per-layer, per-token head-averaged entropy curves (one faint line per token + a bold mean line). **No legend** (it was crowding the figures).
2. A layer-comparison plot: a single head- and token-averaged curve per layer, labeled inline, optionally aggregated over several sentences so it clearly is not a single-example artifact.

Run top to bottom on a T4.

## Part 0 - Environment setup and model loading (from Raghu's notebook)

In [ ]:
!nvidia-smi

!pip install -q transformers==4.44.2 huggingface_hub

!rm -rf DiffuLLaMA && git clone --depth 1 https://github.com/HKUNLP/DiffuLLaMA.git
%cd DiffuLLaMA

import torch
from transformers import AutoConfig, AutoTokenizer
from model import DiscreteDiffusionModel, generate_samples

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL = "gpt2"  # only the config is read from this; weights come from MODEL_NAME

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME,
    model=BASE_MODEL,
    config=config,
    tokenizer=tokenizer,
    device="cuda",
).to("cuda")
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME}: {n_params/1e6:.1f}M params, hidden={config.hidden_size}, layers={config.num_hidden_layers}")


## Forward / tokenization helpers (from Raghu's notebook)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["pdf.fonttype"] = 42  # Embed fonts in PDF outputs for reproducible/publication-friendly figures.
plt.rcParams["ps.fonttype"] = 42
import pandas as pd
from model import get_anneal_attn_mask

@torch.no_grad()
def forward_with_attentions(model, input_ids, attention_mask):
    """
    Runs one DiffuGPT denoising forward pass and returns:
    - logits
    - attentions from every layer

    attentions[layer] shape:
        [batch, num_heads, seq_len, seq_len]
    """
    x_embed = model.get_embeds(input_ids)

    outputs = model.denoise_model(
        inputs_embeds=x_embed,
        attention_mask=attention_mask,
        output_attentions=True,
        output_hidden_states=False,
        return_dict=True,
        use_cache=False,
    )

    logits = model.get_logits(outputs.last_hidden_state)
    attentions = outputs.attentions

    return logits, attentions

text_sequence = "Today is a wonderful day,"

def tokenize_for_experiment(tokenizer, text_sequence, add_bos=True):
    token_ids = tokenizer.encode(text_sequence)

    if add_bos:
        token_ids = [tokenizer.bos_token_id] + token_ids

    input_ids = torch.tensor([token_ids], device=model.device)

    readable_tokens = []
    for tok_id in token_ids:
        readable_tokens.append(tokenizer.decode([tok_id]))

    return input_ids, readable_tokens

true_input_ids, readable_tokens = tokenize_for_experiment(tokenizer, text_sequence)

print("seq_len:", true_input_ids.shape[1])
for i, tok in enumerate(readable_tokens):
    print(i, repr(tok))



## Layer-averaged entropy collector

Same teacher-forced random unmasking schedule as Raghu's `collect_attention_entropy_over_time`, but we keep **all heads**: `attentions[layer][0]` has shape `[heads, S, S]`, we take each token-row's attention entropy per head, then average over heads to get a `[steps, S]` matrix.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from model import get_anneal_attn_mask

plt.rcParams["pdf.fonttype"] = 42   # Type 1 / embedded fonts in PDF outputs
plt.rcParams["ps.fonttype"] = 42

@torch.no_grad()
def collect_layer_avg_entropy_over_time(
    model, tokenizer, text_sequence, layer_idx,
    diffusion_steps=64, seed=42, include_bos=True, normalize_entropy=False,
):
    """
    Returns:
        entropy_matrix: [diffusion_steps, seq_len]  (attention entropy averaged over ALL heads in `layer_idx`)
        readable_tokens, unmask_step
    """
    torch.manual_seed(seed); np.random.seed(seed)

    true_input_ids, readable_tokens = tokenize_for_experiment(tokenizer, text_sequence, add_bos=include_bos)
    true_input_ids = true_input_ids.to(model.device)
    batch_size, seq_len = true_input_ids.shape
    if tokenizer.mask_token_id is None:
        raise ValueError("tokenizer.mask_token_id is None.")

    xt = true_input_ids.clone()
    maskable_mask = torch.ones_like(xt, dtype=torch.bool)
    if include_bos:
        maskable_mask[:, 0] = False
    xt = xt.masked_fill(maskable_mask, tokenizer.mask_token_id)

    unmask_step = [-1] * seq_len
    if include_bos:
        unmask_step[0] = 0

    x_embed = model.get_embeds(xt)
    attention_mask = get_anneal_attn_mask(
        seq_len=seq_len, bsz=batch_size, dtype=x_embed.dtype, device=xt.device, attn_mask_ratio=1.0)

    remaining_mask = maskable_mask.clone()
    per_step_token_entropy = []

    for progress_step in range(diffusion_steps):
        diffusion_t = diffusion_steps - progress_step
        logits, attentions = forward_with_attentions(model, xt, attention_mask)
        if attentions is None:
            raise RuntimeError("No attentions returned; ensure output_attentions=True.")

        attn = attentions[layer_idx][0].detach().float().cpu()   # [heads, seq, seq]
        p = attn.clamp_min(1e-12)
        head_token_entropy = -(p * p.log()).sum(dim=-1)          # [heads, seq]
        token_entropy = head_token_entropy.mean(dim=0)           # [seq]  average over heads
        per_step_token_entropy.append(token_entropy)

        if progress_step == diffusion_steps - 1:
            break

        p_to_unmask = 1.0 / diffusion_t
        reveal_now = remaining_mask & (
            torch.rand_like(remaining_mask, dtype=torch.float, device=model.device) < p_to_unmask)
        xt = xt.clone(); xt[reveal_now] = true_input_ids[reveal_now]
        for pos in reveal_now[0].nonzero(as_tuple=True)[0].tolist():
            if unmask_step[pos] == -1:
                unmask_step[pos] = progress_step + 1
        remaining_mask = remaining_mask & (~reveal_now)

    entropy_matrix = torch.stack(per_step_token_entropy, dim=0)  # [steps, seq]
    if normalize_entropy:
        entropy_matrix = entropy_matrix / np.log(seq_len)
    return entropy_matrix.numpy(), readable_tokens, unmask_step

def plot_layer_avg_entropy_curves(entropy_matrix, readable_tokens, layer_idx,
                                  title=None, skip_bos=True, out_path=None):
    """Per-token head-averaged entropy curves + bold mean curve. No legend."""
    num_steps, seq_len = entropy_matrix.shape
    fig = plt.figure(figsize=(10, 6))
    for ti in range(seq_len):
        if skip_bos and ti == 0:
            continue
        plt.plot(range(num_steps), entropy_matrix[:, ti], linewidth=1.2, alpha=0.55)
    mean_curve = entropy_matrix[:, 1:].mean(axis=1) if skip_bos else entropy_matrix.mean(axis=1)
    plt.plot(range(num_steps), mean_curve, color="black", linewidth=3.0)
    plt.text(num_steps - 1, mean_curve[-1], "  mean", va="center", fontsize=10, fontweight="bold")
    plt.xlabel("Diffusion progress step")
    plt.ylabel("Attention entropy (averaged over all heads)")
    plt.title(title or f"Head-averaged attention entropy over diffusion time | Layer {layer_idx}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, format="pdf", bbox_inches="tight")
    plt.show(); plt.close(fig)

print("layer-averaged entropy tools ready")

## Per-layer, per-token head-averaged curves (Layer 0 vs Layer 10)

Each faint line is one token; the bold black line is the mean over tokens. No legend.

In [ ]:
ENTROPY_SENTENCE = "The student solved the difficult problem correctly."
DIFFUSION_STEPS = 64
SEED = 42
OUT_DIR = "layer_avg_entropy"
import os; os.makedirs(OUT_DIR, exist_ok=True)

for L in [0, 10]:
    em, toks, ust = collect_layer_avg_entropy_over_time(
        model, tokenizer, ENTROPY_SENTENCE, layer_idx=L,
        diffusion_steps=DIFFUSION_STEPS, seed=SEED, normalize_entropy=False)
    plot_layer_avg_entropy_curves(
        em, toks, L,
        title=f"Head-averaged attention entropy over diffusion time | Layer {L} | {ENTROPY_SENTENCE!r}",
        out_path=f"{OUT_DIR}/layer_avg_entropy_L{L}.pdf")
    print(f"Layer {L}: mean entropy step0={em[:,1:].mean(axis=1)[0]:.3f} -> stepLast={em[:,1:].mean(axis=1)[-1]:.3f}")

## Layer comparison: average over all heads AND tokens, per layer

The cleanest view of the claim. Each layer becomes one curve (head- and token-averaged, entropy normalized by `log(seq_len)` so sentences are comparable). Curves are labeled inline at the right edge - **no legend box**. By default it aggregates over several sentences so the trend is not a single-example artifact.

In [ ]:
def layer_mean_entropy_curve(text, layer_idx, diffusion_steps=64, seed=42, skip_bos=True):
    em, toks, ust = collect_layer_avg_entropy_over_time(
        model, tokenizer, text, layer_idx=layer_idx,
        diffusion_steps=diffusion_steps, seed=seed, normalize_entropy=True)
    return em[:, 1:].mean(axis=1) if skip_bos else em.mean(axis=1)

def plot_layer_entropy_comparison(sentences, layers=(0, 5, 10, 11),
                                  diffusion_steps=64, seed=42, out_path=None, title=None):
    """One curve per layer = entropy averaged over all heads, all tokens, and all sentences."""
    if isinstance(sentences, str):
        sentences = [sentences]
    fig = plt.figure(figsize=(10, 6))
    results = {}
    for L in layers:
        curves = [layer_mean_entropy_curve(s, L, diffusion_steps, seed) for s in sentences]
        curve = np.mean(np.stack(curves, axis=0), axis=0)   # average over sentences
        results[L] = curve
        line, = plt.plot(range(len(curve)), curve, linewidth=2.6)
        plt.text(len(curve) - 1, curve[-1], f"  L{L}", va="center",
                 fontsize=11, fontweight="bold", color=line.get_color())
    plt.xlabel("Diffusion progress step")
    plt.ylabel("Mean attention entropy (heads & tokens, normalized)")
    plt.title(title or f"Layer-averaged attention entropy over diffusion time ({len(sentences)} sentences)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, format="pdf", bbox_inches="tight")
    plt.show(); plt.close(fig)
    return results

COMPARISON_SENTENCES = [
    "The student solved the difficult problem correctly.",
    "The doctor gave the patient a careful diagnosis.",
    "Today is a wonderful day,",
    "The cat sat quietly on the warm windowsill.",
    "She carefully painted the old wooden fence yesterday.",
]

# Headline: Layer 0 vs Layer 10
res = plot_layer_entropy_comparison(
    COMPARISON_SENTENCES, layers=(0, 10),
    out_path=f"{OUT_DIR}/layer_entropy_comparison_L0_L10.pdf",
    title="Head-averaged attention entropy: Layer 0 (rises) vs Layer 10 (falls)")
for L, c in res.items():
    print(f"Layer {L}: start={c[0]:.3f}  end={c[-1]:.3f}  delta={c[-1]-c[0]:+.3f}")

# Full depth sweep: ALL 12 layers on one plot (labeled inline, no legend)
plot_layer_entropy_comparison(
    COMPARISON_SENTENCES, layers=tuple(range(12)),
    out_path=f"{OUT_DIR}/layer_entropy_comparison_all12.pdf",
    title="Head-averaged attention entropy by layer (0-11) over diffusion time")

## Download PDFs (Colab)

In [ ]:
import shutil
shutil.make_archive("layer_avg_entropy", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("layer_avg_entropy.zip")
except Exception as exc:
    print("Download helper skipped (local Jupyter? use the file browser):", exc)